In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
    data = f.read()

print("Length of Tiny Shakespear ",len(data))

Length of Tiny Shakespear  1115394


__Here's all the characters in English__

In [4]:
chars = sorted(list(set(data)))
print(''.join(chars))

vocab_size = len(chars)
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
s_to_i = {ch:i for i,ch in enumerate(chars)} # encrypt mapping
i_to_s = {i:ch for i,ch in enumerate(chars)} # decrypt mapping

def encode(s):
    return [s_to_i[c] for c in s]
def decode(i):
    return ''.join([i_to_s[c] for c in i])

print(encode('hii there'))
print(decode(encode('hii there')))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


__Lets now Encode entire tiny Shakespear and store it into torch.tensor__

In [6]:
import torch
encrypted_data = torch.tensor(encode(data), dtype= torch.long)
print(encrypted_data.shape, encrypted_data.dtype)
print(encrypted_data[:100])

c:\Users\sunain\Desktop\GPT\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


__Spliting dataset for Train and Validation data__

In [7]:
n = int(0.9*len(encrypted_data))

train_data = encrypted_data[:n]
test_data = encrypted_data[n:]

In [8]:
block_size = 8
train_data[:block_size+1] 

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

__Data Loader: Batches of Chunks of Data__

In [9]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else test_data

    ix = torch.randint(len(encrypted_data)-block_size, (batch_size,))
    x = torch.stack([encrypted_data[i:i+block_size] for i in ix] )
    y = torch.stack([encrypted_data[i+1:i+block_size+1] for i in ix])
    return x,y

xb, yb = get_batch('train')

print("inputs:")
print(xb)
print("target:")
print(yb)

for b in range(batch_size):
    for t in range(block_size):
        cont = xb[b,:t+1]
        target = yb[b,t]
        print(f"when input is: {cont.tolist()} Target is :{target}")

inputs:
tensor([[59, 52, 49, 47, 52, 42,  1, 40],
        [53, 54, 43, 44, 59, 50,  1, 50],
        [27, 24, 33, 25, 26, 21, 13, 10],
        [47, 41, 43,  1, 53, 60, 43, 56]])
target:
tensor([[52, 49, 47, 52, 42,  1, 40, 56],
        [54, 43, 44, 59, 50,  1, 50, 39],
        [24, 33, 25, 26, 21, 13, 10,  0],
        [41, 43,  1, 53, 60, 43, 56, 58]])
when input is: [59] Target is :52
when input is: [59, 52] Target is :49
when input is: [59, 52, 49] Target is :47
when input is: [59, 52, 49, 47] Target is :52
when input is: [59, 52, 49, 47, 52] Target is :42
when input is: [59, 52, 49, 47, 52, 42] Target is :1
when input is: [59, 52, 49, 47, 52, 42, 1] Target is :40
when input is: [59, 52, 49, 47, 52, 42, 1, 40] Target is :56
when input is: [53] Target is :54
when input is: [53, 54] Target is :43
when input is: [53, 54, 43] Target is :44
when input is: [53, 54, 43, 44] Target is :59
when input is: [53, 54, 43, 44, 59] Target is :50
when input is: [53, 54, 43, 44, 59, 50] Target is :1
wh

__Bigram Language Model__


In [10]:
import torch 
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class bigramLanguageModel(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

    def forward(self, idx, targets = None):
        logits = self.token_embedding_table(idx) # B T C

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):# idx is (B,T) array of indices
        for _ in range(max_new_tokens):
            # get the prediction
            logits, loss = self(idx)
            # focus only on the last step
            logits = logits[:,-1,:]# (B,C)
            # apply softmax for probabilities
            prob = F.softmax(logits, dim= -1)
            # sample from the distribution
            idx_next = torch.multinomial(prob, num_samples=1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = bigramLanguageModel(vocab_size)
logits, loss = m(xb,yb)

print(logits.shape)
print(loss)# Expecting 4.17

idx = torch.zeros((1,1),dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.8389, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [11]:
# create a pytorch optimizer 
optimizer= torch.optim.Adam(m.parameters(), lr=1e-3)

In [12]:
batch_size = 32
for steps in range(10000):
    xb,yb = get_batch('train')

    logits, loss = m(xb,yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.4800143241882324


## Test Bigram Output

In [13]:
idx = torch.zeros((1,1),dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=300)[0].tolist()))



llo br. ave aviasurf my, mayo t ivee ioulrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;

Whe, oraingofoff ve!
RIfans picspeserer hee anf,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
Bin!! Indy; by s afreanoo adicererupa anse tecor


__The Mathematical trick in self-attention__

In [14]:
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [15]:
# version 1  - simple mean averaging
xbow = torch.zeros((B,T,C)) # x bag of words
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev,0)

In [16]:
xbow[2]

tensor([[-0.6631, -0.2513],
        [ 0.1735, -0.0649],
        [ 0.1685,  0.3348],
        [-0.1621,  0.1765],
        [-0.2312, -0.0436],
        [-0.1015, -0.2855],
        [-0.2593, -0.1630],
        [-0.3015, -0.2293]])

In [17]:
# version 2 - matrix multiplication a @ b = c
# instead of prev loop we can use tril 
weights = torch.tril(torch.ones(T,T))
weights = weights / weights.sum(1, keepdim=True)
xbow2 = weights @ x
# weights

In [18]:
# version 3 - use Softmax
tril = torch.tril(torch.ones(T,T))
weights = torch.zeros((T,T))
weights = weights.masked_fill(tril == 0, float('-inf'))
weights = F.softmax(weights, dim= -1)
xbow3 = weights @ x
# weights

In [ ]:
# version 4 - Self Attention
torch.manual_seed(1337)
B,T,C = 4,8,32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C,head_size, bias= False)
query = nn.Linear(C,head_size, bias= False)
value = nn.Linear(C,head_size, bias= False)
k = key(x) 
q = query(x)
weights = q @ k.transpose (-2,-1) # (B,T,16) @ (B,16,T) --> (B,T,T)


tril = torch.tril(torch.ones(T,T))
# weights = torch.zeros((T,T))
weights = weights.masked_fill(tril == 0, float('-inf'))
weights = F.softmax(weights, dim= -1)
v = value(x)
xbow3 = weights @ v
# xbow3 = weights @ x 
xbow3.shape

torch.Size([4, 8, 16])